# Chapter 6 &mdash; Language Equivalence by Lock-Step Search

**Concept 4 of the Chapter 6 decomposition:** *Language Equivalence Checking by Lock-Step Search, with Counterexamples*

Concurrent DFS over state pairs; the first pair whose accept-status disagrees is a counterexample.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6/Concept-Language-Equivalence-Checking/Concept-Language-Equivalence-Checking.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


To decide whether two DFA accept the same language, **walk them together**. Start at
$(q_{0,1}, q_{0,2})$; from each visited pair, follow every symbol in both machines at
once.

* if some reachable pair has **differing finality**, the path that reached it is a
  **counterexample string** &mdash; report it;
* if the search closes with no such pair, the languages are **equal**.

Termination is immediate: there are finitely many pairs. Jove's
`langeq_dfa(D1, D2, gen_counterex=True)` prints the witnessing path.

## 2. Definitions

### Three machines: two equal, one subtly different

In [ ]:
A = md2mc('''DFA
IF : 0 -> Od
IF : 1 -> IF
Od : 0 -> IF
Od : 1 -> Od
''')
B = md2mc('''DFA
IF : 0 -> P
IF : 1 -> Q
P  : 0 -> IF
P  : 1 -> R
Q  : 0 -> R
Q  : 1 -> IF
R  : 0 -> Q
R  : 1 -> P
''')
C = md2mc('''DFA
IF : 0 -> Od
IF : 1 -> Od      !! differs: 1s flip the parity too
Od : 0 -> IF
Od : 1 -> IF
''')

### The lock-step walk, written out, returning the witness string

In [ ]:
def lockstep(D1, D2):
    from collections import deque
    start = (D1["q0"], D2["q0"])
    seen, dq = {start: ''}, deque([start])
    while dq:
        p = dq.popleft(); w = seen[p]
        if (p[0] in D1["F"]) != (p[1] in D2["F"]):
            return w                       # counterexample
        for ch in sorted(D1["Sigma"]):
            n = (step_dfa(D1, p[0], ch), step_dfa(D2, p[1], ch))
            if n not in seen:
                seen[n] = w + ch; dq.append(n)
    return None                            # equivalent

## 3. Tests

Wait &mdash; `B` is not equal to `A`. The walk finds the shortest witness.

In [ ]:
w = lockstep(A, B)
print("witness :", repr(w))
if w is not None:
    print("  A accepts %r ? %s" % (w, accepts_dfa(A, w)))
    print("  B accepts %r ? %s" % (w, accepts_dfa(B, w)))
    assert accepts_dfa(A, w) != accepts_dfa(B, w)
print("langeq_dfa agrees :", langeq_dfa(A, B))
assert (w is None) == langeq_dfa(A, B)

A genuinely equivalent pair closes the search with no witness.

In [ ]:
A2 = md2mc('''DFA
IF : 0 -> X
IF : 1 -> IF
X  : 0 -> Y
X  : 1 -> X
Y  : 0 -> X       !! Y behaves exactly like IF's partner
Y  : 1 -> Y
''')
print("A vs A itself :", lockstep(A, A), langeq_dfa(A, A))
assert lockstep(A, A) is None and langeq_dfa(A, A)

And `C` differs from `A` almost immediately.

In [ ]:
w = lockstep(A, C)
print("witness :", repr(w), " len", len(w))
print("  A:", accepts_dfa(A, w), "  C:", accepts_dfa(C, w))
assert not langeq_dfa(A, C)

Jove prints the visited pairs when you ask for a counterexample.

In [ ]:
print(langeq_dfa(A, C, gen_counterex=True))

Termination: at most $|Q_1|\cdot|Q_2|$ pairs, so the walk always finishes.

In [ ]:
print("pair space for A vs B : %d x %d = %d"
      % (len(A["Q"]), len(B["Q"]), len(A["Q"]) * len(B["Q"])))

## 4. Exercises


1. Modify `lockstep` to return **all** shortest witnesses. How many are there?
2. Why must both DFA be **total** for the walk to be sound?
3. Compare the cost of `langeq_dfa` with "minimize both, then `iso_dfa`".

In [ ]:
# Your work for the exercises above.